# Prompt 5 — adversarial review
Five biggest risks: false identity merges; ambiguous event-to-edge links; missing coverage read as safety; IDs repeating across scenarios; dataset mapping loss. Tests below deliberately exercise each.

In [1]:
from pathlib import Path
import json, sys, hashlib, statistics
from collections import Counter, defaultdict
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'data_model/dataset/attack_path_dataset/data/model_input').is_dir())
sys.path.insert(0,str(ROOT))
DATA = ROOT/'data_model/dataset/attack_path_dataset/data'
MODEL = DATA/'model_input'
AUDIT = ROOT/'phase_a_audit'
ART = AUDIT/'artifacts'
ART.mkdir(parents=True,exist_ok=True)
TWIN_SOURCE_SHA = 'acfc662a54d5fcf02bd7adc896897946e6b35eae19cc500314000633f6c113f5'
checks = []
def check(name, condition):
    checks.append({'name':name,'passed':bool(condition)})
    print(('PASS: ' if condition else 'FAIL: ')+name)
    assert condition, name
def current_inputs():
    return {p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted(MODEL.glob('*')) if p.is_file()}
def save_stage(stage, payload):
    envelope = dict(run_id=run_id, input_hashes=current_inputs(), source_sha=TWIN_SOURCE_SHA, payload=payload)
    (ART/(stage+'.json')).write_text(json.dumps(envelope),encoding='utf-8')
def load_stage(stage):
    envelope = json.loads((ART/(stage+'.json')).read_text(encoding='utf-8'))
    check(stage+' current input fingerprints',envelope['input_hashes']==current_inputs())
    check(stage+' current source fingerprint',envelope['source_sha']==TWIN_SOURCE_SHA)
    check(stage+' current execution run',envelope['run_id']==run_id)
    return envelope['payload']
def report(stage, values):
    values['checks'] = checks.copy()
    (AUDIT/(stage+'_metrics.json')).write_text(json.dumps(values,indent=2),encoding='utf-8')
    print(json.dumps(values,indent=2))
def graph_pack(graphs):
    return {sid:dict(metadata=g.graph,nodes=[[n,d] for n,d in g.nodes(data=True)],edges=[[u,v,k,d] for u,v,k,d in g.edges(keys=True,data=True)]) for sid,g in graphs.items()}
def graph_unpack(packed):
    import networkx as nx
    graphs={}
    for sid,entry in packed.items():
        g=nx.MultiDiGraph()
        g.graph.update(entry['metadata'])
        g.add_nodes_from((n,d) for n,d in entry['nodes'])
        for u,v,k,d in entry['edges']:
            g.add_edge(u,v,key=k,**d)
        graphs[sid]=g
    return graphs

run_id = json.loads((ART/'00_recon.json').read_text())['run_id']

from backend.app.twin.twin import *


## 1. False identity merges

In [2]:
import copy
a = read_rows(MODEL/'assets.jsonl')[0]
a = dict(a, scenario_id='s', asset_id='a', display_name='Same Name')
b = dict(a, asset_id='b')
res = resolve_entity('Same Name', build_alias_index([a,b], 's'), 's')
assert res['confidence']=='low' and res['ambiguous'] and set(res['resolved'])=={'a','b'}
assert a['asset_id'] != b['asset_id']
print('PASS: duplicate display names remain separate low-confidence candidates')

PASS: duplicate display names remain separate low-confidence candidates


## 2. Parallel edges and ambiguous links

In [3]:
s = read_rows(MODEL/'scenarios.jsonl')[0]
sid = s['scenario_id']
assets = read_rows(MODEL/'assets.jsonl')
r = next(r for r in read_rows(MODEL/'relationships.jsonl') if r['scenario_id']==sid)
r2 = dict(r, edge_id='parallel_test')
G = build_potential_graph(sid, assets, [r,r2], s)
ev = dict(event_id='test', scenario_id=sid, event_type='login_success', source='synthetic_idp_logs', ts='2026-01-01T00:00:00Z', confidence=1., actor=r['source_id'], target=r['target_id'])
es,_ = resolve_events([ev],assets)
ls = build_ledger(es,{sid:G})
assert len(ls)==2 and all(x['link']=='pair_match_ambiguous' and x['weight_hint']<1 for x in ls)
assert G.number_of_edges()==2
print('PASS: both parallel candidates survive with lower weights')

PASS: both parallel candidates survive with lower weights


## 3. Blind spots must not change probabilities

In [4]:
H = build_potential_graph(sid,assets,[r],s)
before = [(d['prior_probability'],d['p'],d['variance']) for _,_,d in H.edges(data=True)]
compute_coverage({sid:H},[],[])
assert all(d['coverage']['blind_spot'] and d['coverage']['n_events']==0 for _,_,d in H.edges(data=True))
assert before==[(d['prior_probability'],d['p'],d['variance']) for _,_,d in H.edges(data=True)]
print('PASS: zero evidence does not discount probability')

PASS: zero evidence does not discount probability


## 4. IDs repeating across scenarios

In [5]:
other='other_scenario'
H2 = build_potential_graph(other,[dict(a,scenario_id=other) for a in assets if a['scenario_id']==sid],[dict(r,scenario_id=other)],s)
compute_coverage({sid:H,other:H2},[],[])
build_ledger(es,{sid:H,other:H2})
assert any(d['evidence'] for _,_,d in H.edges(data=True))
assert all(not d['evidence'] for _,_,d in H2.edges(data=True))
assert (sid,r['edge_id']) != (other,r['edge_id'])
print('PASS: repeated edge IDs are isolated')

PASS: repeated edge IDs are isolated


## 5. Mapping loss and contract rejection

In [6]:
gs,es,ls,ms,errs = run_pipeline(MODEL)
for g in gs.values():
    doc = export_graph_json(g)
    assert all(n['dataset_type'] for n in doc['nodes'])
    assert all(e['dataset_type'] for e in doc['edges'])
bad = copy.deepcopy(doc)
del bad['nodes'][0]['dataset_type']
from pydantic import ValidationError
try:
    ScenarioGraphModel.model_validate(bad)
except ValidationError:
    pass
else:
    raise AssertionError('missing dataset_type accepted')
assert ASSET_PLAN_TYPE['user_identity']==ASSET_PLAN_TYPE['cloud_identity']
print('PASS: raw types survive coarse mapping; missing type rejected')

PASS: raw types survive coarse mapping; missing type rejected


## Malformed telemetry and valid null targets

In [7]:
raw = read_rows(MODEL/'events.jsonl')[0]
evidence = {'collection_confidence': .8}
missing = dict(raw); del missing['event_id']
errors=[]
for broken in (missing,dict(raw,timestamp='not-a-date'),dict(raw,event_type='unknown_type')):
    assert normalise_event(broken,evidence,errors) is None
assert len(errors)==3
null_event=normalise_event(dict(raw,target_entity_raw=None),evidence,errors)
assert null_event is not None
resolved,_=resolve_events([null_event],assets)
assert resolved[0]['target_resolved']['confidence']=='unresolved'
print('PASS: malformed rows logged/skipped; null target safely preserved')

PASS: malformed rows logged/skipped; null target safely preserved


## Observed isolation and duplicate coverage

In [8]:
for g in gs.values():
    potential = g.edge_subgraph([(u,v,k) for u,v,k,d in g.edges(keys=True,data=True) if d['layer']=='potential'])
    assert all(d['layer']=='potential' for _,d in potential.nodes(data=True))
    assert all(d.get('p') is None and d.get('variance') is None for _,_,d in g.edges(data=True))
original = copy.deepcopy(ev)
duplicate = dict(copy.deepcopy(ev), event_id='test_duplicate', duplicate_of='test')
small = build_potential_graph(sid, assets, [r], s)
rows = build_ledger([original,duplicate],{sid:small})
compute_coverage({sid:small},[original,duplicate],rows)
assert len(rows)==2 and rows[1]['weight_hint']==0
assert next(iter(small.edges(data=True)))[2]['coverage']['n_events']==1
print('PASS: observed nodes excluded from movement; duplicate coverage is not inflated')

PASS: observed nodes excluded from movement; duplicate coverage is not inflated


## Known limitations
- Permission, API and Cloud Asset nodes and USES, STORES and GRANTS_ACCESS_TO edges remain empty. Secret/EXPOSED_IN are deliberately empty because location categories cannot identify an affected asset; ambiguous scanner events retain only Event.
- Potential plan types currently use coarse legacy vocabulary rather than the handoff's proposed canonical names. These mappings and vulnerability target semantics are plausible (b), not domain-validated. Always retain dataset_type. EXPLOITS in the observed layer means a finding, not demonstrated exploitation.
- Pair links, identity confidence and duplicate detection are heuristics (c). Ambiguous candidates are retained with reduced weight; no physical identity merges occur. Duplicate rows are retained with zero weight and excluded from coverage counts. Weak or missing coverage means a blind spot, never safety.
- Malformed mandatory fields, timestamps and unknown event types are logged and skipped; null targets are valid and retained unresolved. Rejected records need operational quarantine/replay outside this demo.
- All data is synthetic, including invented identities and SYN findings. Evaluation measures fit to this generator, not real-world attack detection or new scenario-family generalization.
- No shared-latent dependency model is available. Shared identity groups are structural hints, not learned correlated-risk estimates. p and variance remain null for the prior owner.
- Temporal causal checks are deferred: no credential-specific prerequisite binding is available. No runtime or throughput benchmark is claimed. Streaming rebuilds prefixes and is a correctness demo, not an incremental production implementation.
- Export contract changes require BGNN/Monte Carlo/API consumer agreement. Consumer integration and production deployment are unverified.


## Final assessment
(a) Verified by the executed cells: scoped parallel edges, soft resolution, duplicate-aware coverage, all-scenario contract validation, malformed-row handling, and guarded inference. Evaluation numbers appear in Prompt 3 outputs.
(b) Plausible: vulnerability target interpretation and coarse mappings. Consumer integration remains unverified.
(c) Guess: candidate weights, confidence scores and near-duplicate thresholds; none are calibrated.
Cut/deferred: credential-bound temporal reasoning, ambiguous Secret placement, shared-latent risk, production incremental streaming and consumer integration.

In [9]:
# These groups count only after their assertion cells above have executed successfully.
groups=['duplicate display names','parallel edge candidates','zero coverage leaves probability unchanged','scenario-scoped repeated IDs','dataset type preservation and schema rejection','malformed events and null target retention','observed isolation and duplicate-aware coverage']
for name in groups:check(name,True)
module_source=(ROOT/'backend/app/twin/twin.py').read_text(encoding='utf-8')
check('Generated module matches audited notebook source',hashlib.sha256(module_source.encode()).hexdigest()==TWIN_SOURCE_SHA)
stage3=load_stage('03_observed')
check('Review pipeline has current ledger output',ls==stage3['ledger'])
limitations = "## Known limitations\n- Permission, API and Cloud Asset nodes and USES, STORES and GRANTS_ACCESS_TO edges remain empty. Secret/EXPOSED_IN are deliberately empty because location categories cannot identify an affected asset; ambiguous scanner events retain only Event.\n- Potential plan types currently use coarse legacy vocabulary rather than the handoff's proposed canonical names. These mappings and vulnerability target semantics are plausible (b), not domain-validated. Always retain dataset_type. EXPLOITS in the observed layer means a finding, not demonstrated exploitation.\n- Pair links, identity confidence and duplicate detection are heuristics (c). Ambiguous candidates are retained with reduced weight; no physical identity merges occur. Duplicate rows are retained with zero weight and excluded from coverage counts. Weak or missing coverage means a blind spot, never safety.\n- Malformed mandatory fields, timestamps and unknown event types are logged and skipped; null targets are valid and retained unresolved. Rejected records need operational quarantine/replay outside this demo.\n- All data is synthetic, including invented identities and SYN findings. Evaluation measures fit to this generator, not real-world attack detection or new scenario-family generalization.\n- No shared-latent dependency model is available. Shared identity groups are structural hints, not learned correlated-risk estimates. p and variance remain null for the prior owner.\n- Temporal causal checks are deferred: no credential-specific prerequisite binding is available. No runtime or throughput benchmark is claimed. Streaming rebuilds prefixes and is a correctness demo, not an incremental production implementation.\n- Export contract changes require BGNN/Monte Carlo/API consumer agreement. Consumer integration and production deployment are unverified.\n"
report('05',dict(adversarial_groups_passed=len(groups),adversarial_groups_failed=0,malformed_rejections=len(errors),expected_rejection_logs=errors,limitations=limitations))
print('All executed adversarial review groups passed.')


PASS: duplicate display names
PASS: parallel edge candidates
PASS: zero coverage leaves probability unchanged
PASS: scenario-scoped repeated IDs
PASS: dataset type preservation and schema rejection
PASS: malformed events and null target retention
PASS: observed isolation and duplicate-aware coverage
PASS: Generated module matches audited notebook source


PASS: 03_observed current input fingerprints
PASS: 03_observed current source fingerprint
PASS: 03_observed current execution run
PASS: Review pipeline has current ledger output
{
  "adversarial_groups_passed": 7,
  "adversarial_groups_failed": 0,
  "malformed_rejections": 3,
  "expected_rejection_logs": [
    {
      "event_id": null,
      "reason": "'event_id'"
    },
    {
      "event_id": "evt_000001",
      "reason": "time data 'not-a-date' does not match format '%Y-%m-%dT%H:%M:%SZ'"
    },
    {
      "event_id": "evt_000001",
      "reason": "unknown event type"
    }
  ],
  "limitations": "## Known limitations\n- Permission, API and Cloud Asset nodes and USES, STORES and GRANTS_ACCESS_TO edges remain empty. Secret/EXPOSED_IN are deliberately empty because location categories cannot identify an affected asset; ambiguous scanner events retain only Event.\n- Potential plan types currently use coarse legacy vocabulary rather than the handoff's proposed canonical names. These mapp